<a href="https://colab.research.google.com/github/Honorine-Kabore/genomic_mosq/blob/main/PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q --no-warn-conflicts malariagen_data

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 22.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.8/215.8 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.7/71.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.9/775.9 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/211.3 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 M

In [ ]:
import malariagen_data
import allel
import numpy as np
import plotly.express as px
import plotly.io as pio
import pandas as pd

In [ ]:
# plotting setup
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.gridspec import GridSpec
import matplotlib_venn as venn

In [ ]:
#Mounting Google Drive
import os
from google.colab import drive
drive.mount("drive")

# make dir
results_dir = "drive/MyDrive/Genomic/"
os.makedirs(results_dir, exist_ok=True)

Mounted at drive


In [ ]:
ag3 = malariagen_data.Ag3(pre=True, results_cache=results_dir)
ag3

<MalariaGEN Ag3 API client>
Storage URL                           : gs://vo_agam_release_master_us_central1/
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16, 3.17
Results cache                         : /content/drive/MyDrive/Genomic
Cohorts analysis                      : 20260120
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.6.0
Client location                       : Utah, United States (Google Cloud us-west3)
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16, 3.17
---
Please note that data are subject to terms of use,
for more information see https://www.malariagen.net/data
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.6.0/Ag3.html

In [ ]:
sets =['3.7', '3.8','3.9', '3.10','3.11']
df_samples = ag3.sample_metadata(sample_sets=sets)
sample_query='country=="Burkina Faso"'


In [ ]:
bf_samples = df_samples.query('country == "Burkina Faso" and year > 2004')

In [24]:
pivot_region_taxon = (
    bf_samples
    .pivot_table(
        index="admin1_name",
        columns="taxon",
        values="sample_id",
        aggfunc="count",
        fill_value=0
    )
)
pivot_region_taxon

taxon,arabiensis,coluzzii,gambiae,gcx4,unassigned
admin1_name,,,,,
Boucle du Mouhoun,14,36,2,0,0
Cascades,8,181,31,41,1
Centre-Ouest,0,0,0,6,0
Centre-Sud,11,20,18,0,0
Est,22,139,40,0,0
Hauts-Bassins,60,292,76,0,0
Plateau Central,4,3,0,6,0
Sahel,10,61,0,0,1


In [25]:
sets_n =['3.11']

In [26]:
new_data=ag3.sample_metadata(sample_sets=sets_n)

In [27]:
samples_new=new_data.query("country == 'Burkina Faso'").groupby(["admin1_name","location",
                                                                 "taxon"]).size()
samples_new

admin1_name        location        taxon     
Boucle du Mouhoun  Nassan          arabiensis     13
                                   coluzzii       32
                                   gambiae         2
Cascades           Degue-Degue     coluzzii        4
                                   gambiae         5
                                   unassigned      1
                   Sideradougou    arabiensis      8
                                   coluzzii       10
                                   gambiae        26
Centre-Sud         Po-Dongo        arabiensis     11
                                   coluzzii       20
                                   gambiae        18
Est                Gama            arabiensis     13
                                   coluzzii       39
                                   gambiae        27
                   Nagare          arabiensis      9
                                   coluzzii      100
                                   gambiae        13
Hauts-Bassins      Bana Village    coluzzii      145
                                   gambiae         5
                   Souroukoudinga  arabiensis      8
                                   coluzzii       66
                                   gambiae        18
Sahel              Ouro-Hesso      arabiensis     10
                                   coluzzii       61
                                   unassigned      1
dtype: int64

In [28]:
samples_new.to_excel("samples_new_2022.xlsx")

In [30]:
sets_old =['3.7', '3.8','3.9', '3.10']

In [31]:
old_data=ag3.sample_metadata(sample_sets=sets_old)

In [32]:
samples_old=old_data.query("country == 'Burkina Faso'").groupby(["admin1_name","location",
                                                                 "taxon"]).size()
samples_old

admin1_name        location         taxon     
Boucle du Mouhoun  Kodougou         arabiensis      1
                                    coluzzii        4
Cascades           Tengrela         coluzzii      167
                                    gcx4           41
Centre-Ouest       Koupela          gcx4            6
Centre-Sud         Monomtenga-B     arabiensis      4
Hauts-Bassins      Bana             coluzzii        2
                   Bana Village     arabiensis      3
                                    coluzzii       46
                                    gambiae         5
                   Pala             arabiensis     34
                                    coluzzii        7
                                    gambiae        24
                   Souroukoudinga   coluzzii        1
                                    gambiae         2
                   Souroukoudingan  arabiensis     15
                                    coluzzii       25
                                    gambiae        22
Plateau Central    Goundry          arabiensis      4
                                    coluzzii        3
                                    gcx4            6
dtype: int64

In [ ]:
samples_old.to_excel("samples_old.xlsx")

In [ ]:
# Create a new cohort for taxon
bf_samples = df_samples.query('country == "Burkina Faso" and year > 2004')
cohort_ta = {}
for ta in bf_samples.taxon.unique():
 for lo in bf_samples.location.unique():
  sample_list = list(bf_samples.query("taxon==@ta").sample_id)
  cohort_ta['An.'+ta] = "sample_id in ['"+ "', '".join(sample_list) + "']"

In [ ]:
#create a new cohort with location, taxon
bf_samples = df_samples.query('country == "Burkina Faso" and year > 2004')
cohort_re = {}
for ta in bf_samples.taxon.unique():
    for re in bf_samples.admin1_name.unique():
        sample_list = list(bf_samples.query("admin1_name==@re and taxon==@ta").sample_id)
        cohort_re[re + "_" + ta[:4]] = "sample_id in ['"+ "', '".join(sample_list) + "']"

# PCA

In [ ]:
#select the region  and chromosome arms 3L
region = "3L:15,000,000-41,000,000"
n_snps = 300000
sample_sets =  ['3.7', '3.8','3.9', '3.10','3.11']
sample_query='country == "Burkina Faso" and year > 2004'

In [ ]:
#The ag3.pca() function actually returns two values, so we need to assign two variables when we run it.
pca_df, evr = ag3.pca(region=region, n_snps=n_snps, sample_sets=sample_sets, sample_query=sample_query)

Compute SNP allele counts:   0%|          | 0/15051 [00:00<?, ?it/s]

Compute biallelic diplotypes:   0%|          | 0/16513 [00:00<?, ?it/s]

In [ ]:
ag3.plot_pca_variance(evr)

In [ ]:
#save image
plt.savefig('drive/MyDrive/Genomepop/variance.png',bbox_inches='tight')

In [34]:
#plot the  pca in the chromosome arms 3L year >2004
pca_3L, evr_bfg = ag3.pca(
    region=region,
    n_snps = 300000,
    sample_sets=sample_sets,
    sample_query='country == "Burkina Faso" and year > 2004',
    min_minor_ac=2,
    max_missing_an=0 )

In [ ]:
pca_3L.to_csv('drive/MyDrive/Genomepop/pca_3L.csv', index=False)

In [37]:
ag3.plot_pca_coords_3d(
    pca_3L,
    color=cohort_ta)

In [36]:
ag3.plot_pca_coords(
    pca_3L, x='PC3', y= 'PC4',
    color="taxon")

In [ ]:
#ag3.plot_pca_coords(pca_df, x="PC3", y="PC5", color=cohort_ta, marker_size=5)

In [ ]:
#plot the  pca in the chromosome arms 3L without gcx4
pca_3L_, evr_bfg = ag3.pca(
    region=region,
    n_snps = 300000,
    sample_sets=sample_sets,
    sample_query='country == "Burkina Faso"' and 'taxon != "gcx4" and year > 2004',
    min_minor_ac=2,
    max_missing_an=0 )

In [ ]:
ag3.plot_pca_coords(
    pca_3L_, x='PC1', y= 'PC2',
    color=cohort_ta)

In [ ]:
ag3.plot_pca_coords(
    pca_3L_, x='PC3', y= 'PC4',
    color=cohort_ta)

In [ ]:
ag3.plot_pca_coords(
    pca_3L, x='PC1', y= 'PC2',
    color=cohort_re)

In [ ]:
#plot the  pca in the chromosome arms 3L  for gambiae samples
pca_3L_gam, evr_bfg = ag3.pca(
    region=region,
    n_snps = 300000,
    sample_sets=sample_sets,
    sample_query='country == "Burkina Faso"' and 'taxon == "gambiae" and year > 2004',
    min_minor_ac=2,
    max_missing_an=0 )

In [ ]:
ag3.plot_pca_coords(
    pca_3L_gam, x='PC1', y= 'PC2',
    color=cohort_re)

In [ ]:
ag3.plot_pca_coords(
    pca_3L_gam, x='PC1', y= 'PC2',
    color="location")

In [ ]:
ag3.plot_pca_coords(
    pca_3L_gam, x='PC1', y= 'PC2',
    color="year")

In [ ]:
#plot the  pca in the chromosome arms 3L for coluzzii samples
pca_3L_col, evr_bfg = ag3.pca(
    region=region,
    n_snps = 300000,
    sample_sets=sample_sets,
    sample_query='country == "Burkina Faso"' and 'taxon == "coluzzii" and year > 2004',
    min_minor_ac=2,
    max_missing_an=0 )

In [ ]:
ag3.plot_pca_coords(
    pca_3L_col, x='PC1', y= 'PC2',
    color=cohort_re)

In [ ]:
ag3.plot_pca_coords(
    pca_3L_col, x='PC1', y= 'PC2',
    color="location")

In [ ]:
ag3.plot_pca_coords(
    pca_3L_col, x='PC1', y= 'PC2',
    color="year")

In [ ]:
#plot the  pca in the chromosome arms 3L  for arabiensis samples
pca_3L_ara, evr_bfg = ag3.pca(
    region=region,
    n_snps = 300000,
    sample_sets=sample_sets,
    sample_query='country == "Burkina Faso"' and 'taxon == "arabiensis" and year > 2004',
    min_minor_ac=2,
    max_missing_an=0 )

In [ ]:
ag3.plot_pca_coords(
    pca_3L_ara, x='PC1', y= 'PC2',
    color=cohort_re)

In [ ]:
ag3.plot_pca_coords(
    pca_3L_ara, x='PC1', y= 'PC2',
    color="location")

In [ ]:
ag3.plot_pca_coords(
    pca_3L_ara, x='PC1', y= 'PC2',
    color="year")

In [ ]:
#plot the  pca in the chromosome  X year >2004
pca_X, evr_bfg = ag3.pca(
    region="X",
    n_snps = 300000,
    sample_sets=sample_sets,
    sample_query='country == "Burkina Faso" and year > 2004',
    min_minor_ac=2,
    max_missing_an=0 )

In [ ]:
ag3.plot_pca_coords(
    pca_X, x='PC1', y= 'PC2',
    color=cohort_re)

In [ ]:
#plot the  pca in the chromosome X  for arabiensis samples
pca_X_ara, evr_bfg = ag3.pca(
    region="X",
    n_snps = 300000,
    sample_sets=sample_sets,
    sample_query='country == "Burkina Faso"' and 'taxon == "arabiensis" and year > 2004',
    min_minor_ac=2,
    max_missing_an=0 )

In [ ]:
ag3.plot_pca_coords(
    pca_X_ara, x='PC1', y= 'PC2',
    color=cohort_re)